# Ensembles

In [5]:
import pandas as pd
from sklearn.ensemble import BaggingRegressor, BaggingClassifier, GradientBoostingRegressor, GradientBoostingClassifier, StackingRegressor, StackingClassifier
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier as KNN
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, r2_score, confusion_matrix, classification_report, make_scorer
from math import sqrt

In [6]:
clsf_data = pd.read_csv('../data/processed_smoke_detector.csv')
X_clsf = clsf_data.drop(['Fire Alarm'], axis=1)
y_clsf = clsf_data['Fire Alarm']
X_clsf_train, X_clsf_test, y_clsf_train, y_clsf_test = train_test_split(X_clsf, y_clsf, test_size=0.2)

In [7]:
regr_data = pd.read_csv('../data/processed_trip_duration.csv')
X_regr = regr_data.drop(['trip_duration'], axis=1)
y_regr = regr_data['trip_duration']
X_regr_train, X_regr_test, y_regr_train, y_regr_test = train_test_split(X_regr, y_regr, test_size=0.2)

## BaggingRegressor


In [8]:
bagging_regr = BaggingRegressor(
    estimator=DecisionTreeRegressor(
        criterion='squared_error',
        splitter= 'best',
        max_depth=15,
        min_samples_split=6,
        min_samples_leaf=7
    ),
    n_estimators=10,
    max_samples=0.8,
    random_state=42,
    n_jobs=-1
)

In [9]:
bagging_regr.fit(X_regr_train, y_regr_train)

BaggingRegressor(estimator=DecisionTreeRegressor(max_depth=15,
                                                 min_samples_leaf=7,
                                                 min_samples_split=6),
                 max_samples=0.8, n_jobs=-1, random_state=42)

In [10]:
y_regr_pred = bagging_regr.predict(X_regr_test)

In [11]:
print(f'MAE: {mean_absolute_error(y_regr_test, y_regr_pred)}')
print(f'MSE: {mean_squared_error(y_regr_test, y_regr_pred)}')
print(f'RMSE: {sqrt(mean_squared_error(y_regr_test, y_regr_pred))}')
print(f'MAPE: {sqrt(mean_absolute_percentage_error(y_regr_test, y_regr_pred))}')
print(f'R^2: {round(r2_score(y_regr_test, y_regr_pred),2)}')

MAE: 193.8910542276041
MSE: 74746.98658968053
RMSE: 273.3989513324448
MAPE: 0.5728877073123642
R^2: 0.68


## BaggingClassifier


In [12]:
bagging_clsf = BaggingClassifier(
    estimator=DecisionTreeClassifier(
        **{'criterion': 'entropy', 'splitter': 'best', 'max_depth': 12, 'min_samples_split': 2, 'min_samples_leaf': 3, 'max_features': 'sqrt'}
    ),
    n_estimators=50,
    max_samples=0.8,
    random_state=42
)

In [13]:
bagging_clsf.fit(X_clsf_train, y_clsf_train)

BaggingClassifier(estimator=DecisionTreeClassifier(criterion='entropy',
                                                   max_depth=12,
                                                   max_features='sqrt',
                                                   min_samples_leaf=3),
                  max_samples=0.8, n_estimators=50, random_state=42)

In [14]:
y_clsf_pred = bagging_clsf.predict(X_clsf_test)

In [15]:
print(confusion_matrix(y_clsf_test, y_clsf_pred))
print(classification_report(y_clsf_test, y_clsf_pred))

[[1924    0]
 [   0 6326]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1924
           1       1.00      1.00      1.00      6326

    accuracy                           1.00      8250
   macro avg       1.00      1.00      1.00      8250
weighted avg       1.00      1.00      1.00      8250



## GradientBoostingRegressor


In [16]:
# param_grid = {
#     'loss': ['squared_error', 'huber'],
#     'learning_rate': [0.05, 0.1],
#     'subsample': [0.8, 1.0],
#     'criterion': ['friedman_mse', 'squared_error'],
#     'min_samples_split': [2, 5],
#     'min_samples_leaf': [1, 3],
#     'max_depth': [3, 5],
#     'max_features': [None, 'sqrt'],
#     'alpha': [0.85, 0.9]
# }

# fixed_params = {
#     'n_estimators': 100,
#     'random_state': 42,
#     'min_weight_fraction_leaf': 0,
#     'min_impurity_decrease': 0,
#     'init': None
# }

# gb = GradientBoostingRegressor(**fixed_params)

# grid_search = GridSearchCV(
#     estimator=gb,
#     param_grid=param_grid,
#     scoring=make_scorer(mean_squared_error, greater_is_better=False),
#     cv=5,
#     n_jobs=-1,
#     verbose=2
# )

# grid_search.fit(X_regr_train, y_regr_train)

# print(f"Лучшие параметры: {grid_search.best_params_}")
# print(f"Лучший MSE: {-grid_search.best_score_:.4f}")

In [17]:
gb_regr = GradientBoostingRegressor(
    n_estimators=20,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

In [18]:
gb_regr.fit(X_regr_train, y_regr_train)

GradientBoostingRegressor(n_estimators=20, random_state=42)

In [19]:
y_regr_pred = gb_regr.predict(X_regr_test)

In [20]:
print(f'MAE: {mean_absolute_error(y_regr_test, y_regr_pred)}')
print(f'MSE: {mean_squared_error(y_regr_test, y_regr_pred)}')
print(f'RMSE: {sqrt(mean_squared_error(y_regr_test, y_regr_pred))}')
print(f'MAPE: {sqrt(mean_absolute_percentage_error(y_regr_test, y_regr_pred))}')
print(f'R^2: {round(r2_score(y_regr_test, y_regr_pred),2)}')

MAE: 233.1772605773402
MSE: 99847.58340812371
RMSE: 315.9866823271571
MAPE: 0.6574492805761735
R^2: 0.57


## GradientBoostingClassifier


In [21]:
gb_clsf = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

In [22]:
gb_clsf.fit(X_clsf_train, y_clsf_train)

GradientBoostingClassifier(random_state=42)

In [23]:
y_clsf_pred = gb_clsf.predict(X_clsf_test)

In [24]:
print(confusion_matrix(y_clsf_test, y_clsf_pred))
print(classification_report(y_clsf_test, y_clsf_pred))

[[1924    0]
 [   1 6325]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1924
           1       1.00      1.00      1.00      6326

    accuracy                           1.00      8250
   macro avg       1.00      1.00      1.00      8250
weighted avg       1.00      1.00      1.00      8250



## StackingRegressor


In [25]:
estimators = [
    ('dt', DecisionTreeRegressor(max_depth=4)),
    ('ridge', Ridge())
]

In [26]:

stacking_reg = StackingRegressor(
    estimators=estimators,
    final_estimator=LinearRegression(),
    n_jobs=-1
)

In [27]:
stacking_reg.fit(X_regr_train, y_regr_train)

StackingRegressor(estimators=[('dt', DecisionTreeRegressor(max_depth=4)),
                              ('ridge', Ridge())],
                  final_estimator=LinearRegression(), n_jobs=-1)

In [28]:
y_pred = stacking_reg.predict(X_regr_test)

In [29]:
print(f'MAE: {mean_absolute_error(y_regr_test, y_regr_pred)}')
print(f'MSE: {mean_squared_error(y_regr_test, y_regr_pred)}')
print(f'RMSE: {sqrt(mean_squared_error(y_regr_test, y_regr_pred))}')
print(f'MAPE: {sqrt(mean_absolute_percentage_error(y_regr_test, y_regr_pred))}')
print(f'R^2: {round(r2_score(y_regr_test, y_regr_pred),2)}')

MAE: 233.1772605773402
MSE: 99847.58340812371
RMSE: 315.9866823271571
MAPE: 0.6574492805761735
R^2: 0.57


## StackingClassifier

In [32]:
estimators = [
    ('dt', DecisionTreeClassifier()),
    ('svc', KNN())
]

In [33]:
stacking_clf = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(),
    n_jobs=-1   
)

In [34]:
stacking_clf.fit(X_clsf_train, y_clsf_train)

StackingClassifier(estimators=[('dt', DecisionTreeClassifier()),
                               ('svc', KNeighborsClassifier())],
                   final_estimator=LogisticRegression(), n_jobs=-1)

In [35]:
y_clsf_pred = stacking_clf.predict(X_clsf_test)

In [36]:
print(confusion_matrix(y_clsf_test, y_clsf_pred))
print(classification_report(y_clsf_test, y_clsf_pred))

[[1924    0]
 [   0 6326]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1924
           1       1.00      1.00      1.00      6326

    accuracy                           1.00      8250
   macro avg       1.00      1.00      1.00      8250
weighted avg       1.00      1.00      1.00      8250

